In [ ]:
import time
# Let starter.py compute 12 hours from its own start
global_end_time = 0.0

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    if not (isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)):
        return False
    if np.any((guess < 0) | (guess > 9)):   # <-- যোগ করুন
        return False
    return True
    
def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    if sorted(permutation) != list(range(10)):   # <-- assert-এর পরিবর্তে
        raise ValueError(f"Invalid permutation descriptor: {descriptor}")
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))
    @staticmethod
    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not query['train']:   # <-- যোগ করুন
                    break
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            num_guesses = len(v)
            for i in range(len(target_dict)):  # সাধারণত 2
                if i < num_guesses:
                    target_dict[f'attempt_{i+1}'] = v[i].tolist()
                else:
                    # শেষ guess-টি বারবার ব্যবহার করুন
                    target_dict[f'attempt_{i+1}'] = v[-1].tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs



def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def run_dual_selection(self):
        """Return {base_key: [best_from_probmul, best_from_kgmon]}"""
        from arc_decoder import score_full_probmul_3, score_kgmon
        results = {}
        for bk, guesses in self.decoded_results.items():
            g = {k: v for k, v in guesses.items()}
            ranked_a = score_full_probmul_3(g)
            ranked_b = score_kgmon(g)
            top_a = ranked_a[0] if ranked_a else None
            top_b = None
            for cand in ranked_b:
                if top_a is None or not np.array_equal(cand, top_a):
                    top_b = cand
                    break
            if top_b is None:
                top_b = top_a
            combined = [x for x in [top_a, top_b] if x is not None]
            results[bk] = combined
        return results
        
    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.rsplit("_", 1)
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        else:
            print(" avg correct beam score: N/A (no correct solutions found)")
            print(" max correct beam score: N/A (no correct solutions found)")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.rsplit("_", 1)[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import copy
import queue
import torch
import numpy as np
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

# ------------------------------------------------------------
# Fixed token IDs (as used in fine-tuning)
# ------------------------------------------------------------
ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

MAX_BEAM_CANDIDATES = 5
class UnslothFixedTrainer(UnslothTrainer):
    # Issue https://github.com/unslothai/unsloth/issues/2435
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            # Ensure labels is 1D (avoid scalar tensor)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            # Use torch.nonzero instead of torch.where to avoid bool scalar issues
            user_mask = (labels == USER_TOKEN_ID)
            assistant_mask = (labels == ASSISTANT_TOKEN_ID)
            user_start_idx = torch.nonzero(user_mask, as_tuple=True)[0].tolist()
            assistant_start_idx = torch.nonzero(assistant_mask, as_tuple=True)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            eos_mask = (labels == EOS_ID)
            end_idx = torch.nonzero(eos_mask, as_tuple=True)[0].tolist()
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                if start >= end:
                    continue
                if j % 2 == 1:  # assistant response part
                    start += 2
                    end += 1
                    if start < end and end <= labels.size(0):
                        batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i].sort(key=lambda x: x[0])
        candidates[i] = candidates[i][:MAX_BEAM_CANDIDATES]

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000.0)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break


        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,    # ← original cache (no crop)
            return_dict=True,
            use_cache=True,
        )

        next_cache = outputs.past_key_values

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=next_cache,   # ← new cache with length pos+1
            start_time=start_time,
            end_time=end_time,
        )

        del outputs
        torch.cuda.empty_cache()

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x: x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    
    max_len = max(batch_lengths)
    padded_tokens = []
    attention_masks = []
    for tokens in batch_tokens:
        pad_len = max_len - len(tokens)
        padded = tokens + [PAD_ID] * pad_len
        padded_tokens.append(padded)
        attention_masks.append([1] * len(tokens) + [0] * pad_len)
        
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    attention_mask = torch.tensor(attention_masks, device=model.device, dtype=torch.long)
    
    # FIX: use_cache=False because padding breaks KV cache
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, use_cache=False)
    
    result = []
    # FIX: Slicing on GPU before transferring to CPU to prevent massive OOM Crash
    for i, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_logits = outputs.logits[i, query_length-1 : query_length-1+len(answer_tokens), :]
        answer_log_probs = answer_logits.float().log_softmax(-1).cpu()
        answer_score = answer_log_probs[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
        
    return result


def worker(rank, p_queue, end_time):
    # CUDA optimizations (from winning solution)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch_fused",  # Fused optimizer for speed
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
        dtype=torch.bfloat16,            # Force bfloat16
        attn_implementation="sdpa",      # Use SDPA (fast, memory efficient)
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    # Fixed queue race condition with Timeout
    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        try:
            key = p_queue.get(timeout=60)
        except queue.Empty:
            break
            
        if key is None:
            break

        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        set_peft_model_state_dict(model, default_weights.copy(), adapter_name="default")
        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        # Puzzle-specific seed for diversity
        train_seed = hash(key) % 10000
        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=train_seed)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )
            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer

        model = FastLanguageModel.for_inference(model)

        gc.collect()
        torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")
        torch.cuda.reset_peak_memory_stats()
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        # FIX: Use puzzle-specific seed for evaluation augmentation
        eval_seed = hash(key) % 10000
        eval_ds = puzzle_ds_multi.augment(n=2, seed=eval_seed)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
            known_scores = {}

            for subkeys in batches:
                spend_time = time.time() - start_time
                if spend_time > 1450 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(
                    model, tokens, max_new_tokens, max_score, end_time
                )

                for subkey_id, scored_beams in dfs_result:
                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:
                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_seed = hash(bk) % 1024**2
                            aug_dataset = aug_dataset.augment(seed=aug_seed)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores

                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if decoded_result:
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import numpy as np
import torch.multiprocessing as mp

# ──────────────────────────────────────────────────────────────
#   Debug mode: puzzle manual run
#   Kaggle submit (rerun) এ automatically all puzzle
# ──────────────────────────────────────────────────────────────
DEBUG_PUZZLES = [
    "0934a4d8",  # tier=3, fine=1.7619, tags=tier3,dynamic,rare,high_ratio,extreme_ratio,many_colors,large,dense_non_bg
    "20a9e565",  # tier=3, fine=1.5125, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,many_colors,large,sparse_bg
    "4c7dc4dd",  # tier=3, fine=1.8181, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,large,dense_non_bg,low_train
    "89565ca0",  # tier=3, fine=1.4844, tags=tier3,dynamic,high_ratio,extreme_ratio,many_colors,large
    "136b0064",  # tier=1, fine=0.4736, tags=tier1,hard_tier1,dynamic,rare,sparse_bg
    "291dc1e1",  # tier=1, fine=0.5169, tags=tier1,hard_tier1,dynamic,rare,many_colors,dense_non_bg
    "16b78196",  # tier=2, fine=0.4400, tags=tier2,rare,large,sparse_bg,low_train
    "271d71e2",  # tier=1, fine=0.1552, tags=tier1,easy_static,easy,rare,dense_non_bg
    "28a6681f",  # tier=1, fine=0.1733, tags=tier1,easy,rare
    "2b83f449",  # tier=1, fine=0.1705, tags=tier1,easy_static,easy,rare,low_train
    "2c181942",  # tier=2, fine=0.2887, tags=tier2,rare,dense_non_bg
    "31f7f899",  # tier=1, fine=0.2092, tags=tier1,easy,rare,many_colors,dense_non_bg
    "35ab12c3",  # tier=1, fine=0.2654, tags=tier1,rare,many_colors,sparse_bg
    "3dc255db",  # tier=1, fine=0.1631, tags=tier1,easy,rare,sparse_bg
    "409aa875",  # tier=1, fine=0.1853, tags=tier1,easy_static,easy,rare,dense_non_bg
    "4c416de3",  # tier=2, fine=0.3368, tags=tier2,rare,many_colors,dense_non_bg
    "64efde09",  # tier=2, fine=0.3802, tags=tier2,rare,many_colors,large,dense_non_bg,low_train
    "6e453dd6",  # tier=1, fine=0.1484, tags=tier1,easy_static,easy,rare,dense_non_bg
    "7491f3cf",  # tier=1, fine=0.2583, tags=tier1,rare,many_colors,dense_non_bg
    "7b0280bc",  # tier=1, fine=0.3054, tags=tier1,rare,many_colors,dense_non_bg
    "7b80bb43",  # tier=2, fine=0.2720, tags=tier2,rare,large,low_train
    "7c66cb00",  # tier=2, fine=0.3800, tags=tier2,rare,large,dense_non_bg
    "1818057f",  # tier=1, fine=0.1177, tags=tier1,easy_static,easy,dense_non_bg
    "332f06d7",  # tier=1, fine=0.1380, tags=tier1,easy_static,easy,dense_non_bg
    "53fb4810",  # tier=1, fine=0.2058, tags=tier1,easy_static,easy,dense_non_bg,low_train
    "7666fa5d",  # tier=1, fine=0.1653, tags=tier1,easy_static,easy,dense_non_bg,low_train
    "da515329",  # tier=1, fine=0.1382, tags=tier1,easy_static,easy,sparse_bg
    "8f3a5a89",  # tier=1, fine=0.1640, tags=tier1,easy_static,easy,rare,dense_non_bg
    "db695cfb",  # tier=1, fine=0.1465, tags=tier1,easy_static,easy,rare,dense_non_bg
    "97d7923e",  # tier=1, fine=0.1644, tags=tier1,easy,sparse_bg
    "de809cff",  # tier=1, fine=0.2533, tags=tier1,dense_non_bg,low_train
    "c7f57c3e",  # tier=1, fine=0.2280, tags=tier1,rare,dense_non_bg,low_train
    "cb2d8a2c",  # tier=1, fine=0.1766, tags=tier1,easy_static,easy,multi_query,dense_non_bg
    "58490d8a",  # tier=2, fine=0.7666, tags=tier2,dynamic,high_ratio,dense_non_bg
    "dbff022c",  # tier=1, fine=0.2338, tags=tier1,rare,many_colors
    "5961cc34",  # tier=2, fine=0.2485, tags=tier2,dense_non_bg
    "80a900e0",  # tier=2, fine=0.2867, tags=tier2,large,low_train
    "269e22fb",  # tier=1, fine=0.4397, tags=tier1,dynamic,multi_query,dense_non_bg
    "8b9c3697",  # tier=2, fine=0.2593, tags=tier2,dense_non_bg
    "e376de54",  # tier=1, fine=0.2253, tags=tier1,rare,dense_non_bg
    "b99e7126",  # tier=2, fine=0.3803, tags=tier2,large,dense_non_bg
    "c4d067a0",  # tier=2, fine=0.2763, tags=tier2,large,dense_non_bg
    "135a2760",  # tier=1, fine=0.2078, tags=tier1,easy,rare,dense_non_bg,low_train
    "4a21e3da",  # tier=1, fine=0.1680, tags=tier1,easy_static,easy,multi_query,dense_non_bg
    "71e489b6",  # tier=1, fine=0.1528, tags=tier1,easy_static,easy,multi_query
    "800d221b",  # tier=1, fine=0.2840, tags=tier1,rare,many_colors,dense_non_bg
    "dd6b8c4b",  # tier=1, fine=0.1403, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
    "36a08778",  # tier=1, fine=0.0788, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
]


def get_puzzle_priority(key, data):
    train   = data[key]['train']
    n_train = len(train)

    avg_input_size = np.mean([
        np.prod(np.shape(x['input'])) for x in train
    ])
    size_ratios = [
        max(np.prod(np.shape(x['output'])),
            np.prod(np.shape(x['input']))) /
        max(min(np.prod(np.shape(x['output'])),
                np.prod(np.shape(x['input']))), 1)
        for x in train
    ]
    max_ratio = max(size_ratios)
    avg_ratio = np.mean(size_ratios)

    n_colors = len(set(
        int(c)
        for x in train
        for arr in [x['input'], x['output']]
        for c in np.unique(arr).tolist()
    ))
    is_dynamic = any(
        np.shape(x['input']) != np.shape(x['output'])
        for x in train
    )

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900  * 0.30 +
        (avg_ratio - 1) / 10  * 0.25 +
        n_colors / 10         * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )

    if n_train >= 6 and tier < 3:
        fine *= 0.7

    return (tier, fine)


def local_worker(rank, queue, end_time):
    os.environ["CUDA_VISIBLE_DEVICES"]       = str(rank)
    os.environ["UNSLOTH_DISABLE_RL_PATCH"]   = "1"
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)

    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")
    worker(rank, queue, end_time)
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0,
                        help="Unix timestamp when execution must stop. If 0, compute 12 hours from now.")
    args = parser.parse_args()

    if args.end_time == 0.0:
        end_time = time.time() + 12 * 3600
    else:
        end_time = args.end_time

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    # ── EDA-based ordering ───────────────────────────────
    keys_sorted = sorted(
        data.keys(),
        key=lambda k: get_puzzle_priority(k, data)
    )

    # Tier breakdown log
    tiers = [get_puzzle_priority(k, data)[0] for k in keys_sorted]
    print(f"[Starter] Puzzle tiers — "
          f"Easy: {tiers.count(1)}, "
          f"Hard: {tiers.count(2)}, "
          f"Extreme: {tiers.count(3)}")
    print(f"[Starter] First: {keys_sorted[0]}, "
          f"Last: {keys_sorted[-1]}")

    
    for key in keys_sorted:
        if not rerun_mode:
            if key not in DEBUG_PUZZLES:
                continue
        queue.put(key)

    for _ in range(4):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, end_time), nprocs=4)

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

submission = data.get_submission(decoder.run_dual_selection())

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))

In [ ]:
if decoder.decoded_results:
    test_key = list(decoder.decoded_results.keys())[0]
    result = decoder.run_dual_selection()
    print(f"Key: {test_key}")
    print(f"Attempt 1 shape: {result[test_key][0].shape if result[test_key] else None}")
    print(f"Attempt 2 shape: {result[test_key][1].shape if len(result[test_key]) > 1 else None}")
else:
    print("কোনো রেজাল্ট জেনারেট হয়নি। মডেল ইনফারেন্স সম্ভবত ফেইল করেছে।")

In [ ]:
# ================================================================
# DEBUG PUZZLES - MAIN_DEBUG_48_EVAL_PROXY DIAGNOSTIC
# ================================================================
import hashlib
import re
import numpy as np
from starter import get_puzzle_priority
from arc_decoder import score_full_probmul_3, score_kgmon

STATUS_SOLVED = "SOLVED"
STATUS_UNSOLVED = "UNSOLVED"
STATUS_NO_OUTPUT = "NO OUTPUT"

def deterministic_seed(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % 10000

def status_text(row):
    return str(row.get("status", "")).upper()

def is_no_output(row):
    return status_text(row) == STATUS_NO_OUTPUT

def is_unsolved(row):
    return status_text(row) == STATUS_UNSOLVED

def is_solved(row):
    return status_text(row) == STATUS_SOLVED

def dual_yes(row):
    return "YES" in str(row.get("solved_by_dual", "")).upper()

if hasattr(data, "queries"):
    data_dict = data.queries
else:
    data_dict = data

DEBUG_PUZZLES = [
    "0934a4d8",  # tier=3, fine=1.7619, tags=tier3,dynamic,rare,high_ratio,extreme_ratio,many_colors,large,dense_non_bg
    "20a9e565",  # tier=3, fine=1.5125, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,many_colors,large,sparse_bg
    "4c7dc4dd",  # tier=3, fine=1.8181, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,large,dense_non_bg,low_train
    "89565ca0",  # tier=3, fine=1.4844, tags=tier3,dynamic,high_ratio,extreme_ratio,many_colors,large
    "136b0064",  # tier=1, fine=0.4736, tags=tier1,hard_tier1,dynamic,rare,sparse_bg
    "291dc1e1",  # tier=1, fine=0.5169, tags=tier1,hard_tier1,dynamic,rare,many_colors,dense_non_bg
    "16b78196",  # tier=2, fine=0.4400, tags=tier2,rare,large,sparse_bg,low_train
    "271d71e2",  # tier=1, fine=0.1552, tags=tier1,easy_static,easy,rare,dense_non_bg
    "28a6681f",  # tier=1, fine=0.1733, tags=tier1,easy,rare
    "2b83f449",  # tier=1, fine=0.1705, tags=tier1,easy_static,easy,rare,low_train
    "2c181942",  # tier=2, fine=0.2887, tags=tier2,rare,dense_non_bg
    "31f7f899",  # tier=1, fine=0.2092, tags=tier1,easy,rare,many_colors,dense_non_bg
    "35ab12c3",  # tier=1, fine=0.2654, tags=tier1,rare,many_colors,sparse_bg
    "3dc255db",  # tier=1, fine=0.1631, tags=tier1,easy,rare,sparse_bg
    "409aa875",  # tier=1, fine=0.1853, tags=tier1,easy_static,easy,rare,dense_non_bg
    "4c416de3",  # tier=2, fine=0.3368, tags=tier2,rare,many_colors,dense_non_bg
    "64efde09",  # tier=2, fine=0.3802, tags=tier2,rare,many_colors,large,dense_non_bg,low_train
    "6e453dd6",  # tier=1, fine=0.1484, tags=tier1,easy_static,easy,rare,dense_non_bg
    "7491f3cf",  # tier=1, fine=0.2583, tags=tier1,rare,many_colors,dense_non_bg
    "7b0280bc",  # tier=1, fine=0.3054, tags=tier1,rare,many_colors,dense_non_bg
    "7b80bb43",  # tier=2, fine=0.2720, tags=tier2,rare,large,low_train
    "7c66cb00",  # tier=2, fine=0.3800, tags=tier2,rare,large,dense_non_bg
    "1818057f",  # tier=1, fine=0.1177, tags=tier1,easy_static,easy,dense_non_bg
    "332f06d7",  # tier=1, fine=0.1380, tags=tier1,easy_static,easy,dense_non_bg
    "53fb4810",  # tier=1, fine=0.2058, tags=tier1,easy_static,easy,dense_non_bg,low_train
    "7666fa5d",  # tier=1, fine=0.1653, tags=tier1,easy_static,easy,dense_non_bg,low_train
    "da515329",  # tier=1, fine=0.1382, tags=tier1,easy_static,easy,sparse_bg
    "8f3a5a89",  # tier=1, fine=0.1640, tags=tier1,easy_static,easy,rare,dense_non_bg
    "db695cfb",  # tier=1, fine=0.1465, tags=tier1,easy_static,easy,rare,dense_non_bg
    "97d7923e",  # tier=1, fine=0.1644, tags=tier1,easy,sparse_bg
    "de809cff",  # tier=1, fine=0.2533, tags=tier1,dense_non_bg,low_train
    "c7f57c3e",  # tier=1, fine=0.2280, tags=tier1,rare,dense_non_bg,low_train
    "cb2d8a2c",  # tier=1, fine=0.1766, tags=tier1,easy_static,easy,multi_query,dense_non_bg
    "58490d8a",  # tier=2, fine=0.7666, tags=tier2,dynamic,high_ratio,dense_non_bg
    "dbff022c",  # tier=1, fine=0.2338, tags=tier1,rare,many_colors
    "5961cc34",  # tier=2, fine=0.2485, tags=tier2,dense_non_bg
    "80a900e0",  # tier=2, fine=0.2867, tags=tier2,large,low_train
    "269e22fb",  # tier=1, fine=0.4397, tags=tier1,dynamic,multi_query,dense_non_bg
    "8b9c3697",  # tier=2, fine=0.2593, tags=tier2,dense_non_bg
    "e376de54",  # tier=1, fine=0.2253, tags=tier1,rare,dense_non_bg
    "b99e7126",  # tier=2, fine=0.3803, tags=tier2,large,dense_non_bg
    "c4d067a0",  # tier=2, fine=0.2763, tags=tier2,large,dense_non_bg
    "135a2760",  # tier=1, fine=0.2078, tags=tier1,easy,rare,dense_non_bg,low_train
    "4a21e3da",  # tier=1, fine=0.1680, tags=tier1,easy_static,easy,multi_query,dense_non_bg
    "71e489b6",  # tier=1, fine=0.1528, tags=tier1,easy_static,easy,multi_query
    "800d221b",  # tier=1, fine=0.2840, tags=tier1,rare,many_colors,dense_non_bg
    "dd6b8c4b",  # tier=1, fine=0.1403, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
    "36a08778",  # tier=1, fine=0.0788, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
]

print("=" * 100)
print("CHECK 1: SEED DETERMINISM")
print("=" * 100)
for k in DEBUG_PUZZLES:
    ts = deterministic_seed(k)
    es = deterministic_seed(f"{k}_eval")
    print(f"{k}: train_seed={ts}, eval_seed={es}")
print("(These values must stay identical across every run/commit.)")

print("\n" + "=" * 100)
print("CHECK 2: PUZZLE ORDERING / TIER ASSIGNMENT")
print("=" * 100)

tier_counts = {1: 0, 2: 0, 3: 0}
for k in DEBUG_PUZZLES:
    if k in data_dict:
        tier, fine = get_puzzle_priority(k, data_dict)
        tier_counts[tier] = tier_counts.get(tier, 0) + 1
        print(f"{k}: tier={tier}, fine_score={fine:.4f}")
    else:
        print(f"{k}: NOT FOUND in current dataset")

print(
    f"\nTier distribution: "
    f"Tier1={tier_counts.get(1, 0)}, "
    f"Tier2={tier_counts.get(2, 0)}, "
    f"Tier3={tier_counts.get(3, 0)}"
)

print("\n" + "=" * 100)
print("CHECK 3: TRAIN/EVAL SEED SEPARATION")
print("=" * 100)

all_separated = True
for k in DEBUG_PUZZLES:
    ts = deterministic_seed(k)
    es = deterministic_seed(f"{k}_eval")
    if ts == es:
        all_separated = False
        print(f"BAD {k}: train_seed={ts} == eval_seed={es}")
    else:
        print(f"OK  {k}: train_seed={ts} != eval_seed={es}")

print("\nOK: all train/eval seeds are separated." if all_separated else "\nBAD: some seeds are not separated.")

print("\n" + "=" * 100)
print("CHECK 4: SOLVED / UNSOLVED SUMMARY")
print("=" * 100)

summary_data = []

for puzzle_id in DEBUG_PUZZLES:
    base_keys = [k for k in decoder.decoded_results.keys() if k.startswith(puzzle_id)]

    if not base_keys:
        summary_data.append({
            "id": puzzle_id,
            "status": STATUS_NO_OUTPUT,
            "solved_by_dual": "No",
            "rank_probmul": "N/A",
            "rank_kgmon": "N/A",
            "best_beam": "N/A",
            "n_queries": 0,
        })
        continue

    solved_flag = False
    solved_by_dual = False
    best_beam = None
    rank_probmul = None
    rank_kgmon = None
    n_queries = len(base_keys)

    for bk in base_keys:
        guesses = dict(decoder.decoded_results.get(bk, {}))
        correct_solution = decoder.dataset.replies.get(bk, [None])[0]

        if correct_solution is None or not guesses:
            continue

        correct_found = False
        for g in guesses.values():
            if np.array_equal(g["solution"], correct_solution):
                correct_found = True
                if best_beam is None or g["beam_score"] < best_beam:
                    best_beam = g["beam_score"]
                break

        if correct_found:
            solved_flag = True

            ranked_a = score_full_probmul_3(guesses)
            ranked_b = score_kgmon(guesses)

            local_rank_probmul = None
            local_rank_kgmon = None

            for i, cand in enumerate(ranked_a):
                if np.array_equal(cand, correct_solution):
                    local_rank_probmul = i + 1
                    break

            for i, cand in enumerate(ranked_b):
                if np.array_equal(cand, correct_solution):
                    local_rank_kgmon = i + 1
                    break

            if local_rank_probmul is not None:
                rank_probmul = local_rank_probmul if rank_probmul is None else min(rank_probmul, local_rank_probmul)
            if local_rank_kgmon is not None:
                rank_kgmon = local_rank_kgmon if rank_kgmon is None else min(rank_kgmon, local_rank_kgmon)

            if local_rank_probmul == 1 or local_rank_kgmon == 1:
                solved_by_dual = True

    summary_data.append({
        "id": puzzle_id,
        "status": STATUS_SOLVED if solved_flag else STATUS_UNSOLVED,
        "solved_by_dual": "Yes" if solved_by_dual else "No",
        "rank_probmul": rank_probmul if rank_probmul is not None else "N/A",
        "rank_kgmon": rank_kgmon if rank_kgmon is not None else "N/A",
        "best_beam": f"{best_beam:.5f}" if best_beam is not None else "N/A",
        "n_queries": n_queries,
    })

print(f"\n{'Puzzle ID':<15} {'Status':<12} {'Dual':<8} {'Rank Probmul':<14} {'Rank Kgmon':<12} {'Best Beam':<12} {'Queries':<7}")
print("-" * 90)

for row in summary_data:
    print(
        f"{row['id']:<15} {row['status']:<12} {row['solved_by_dual']:<8} "
        f"{str(row['rank_probmul']):<14} {str(row['rank_kgmon']):<12} "
        f"{row['best_beam']:<12} {row['n_queries']:<7}"
    )

total = len(summary_data)
solved_count = sum(1 for r in summary_data if is_solved(r))
dual_solved = sum(1 for r in summary_data if dual_yes(r))

print("\n" + "=" * 100)
print(f"SUMMARY: {solved_count}/{total} puzzles solved in decoded candidates.")
print(f"         {dual_solved}/{total} puzzles solved by dual selection.")
print("=" * 100)

print("\n" + "=" * 100)
print("CHECK 5: TIMEOUT / MISSING OUTPUT BREAKDOWN")
print("=" * 100)

no_output = [r["id"] for r in summary_data if is_no_output(r)]
unsolved = [r["id"] for r in summary_data if is_unsolved(r)]
solved = [r["id"] for r in summary_data if is_solved(r)]

print(f"No output: {len(no_output)} -> {no_output}")
print(f"Unsolved : {len(unsolved)} -> {unsolved}")
print(f"Solved   : {len(solved)} -> {solved}")
print("NO OUTPUT = infrastructure/timeout/queue issue.")
print("UNSOLVED  = ran, but wrong answer/model issue.")

print("\n" + "=" * 100)
print("CHECK 6: MULTI-QUERY PUZZLE BREAKDOWN")
print("=" * 100)

multi_query = [r for r in summary_data if r["n_queries"] > 1]
if multi_query:
    print(f"Multi-query puzzles found: {len(multi_query)}")
    for r in multi_query:
        print(f"  {r['id']}: {r['n_queries']} queries, status={r['status']}, dual={r['solved_by_dual']}")
else:
    print("No multi-query puzzles in DEBUG_PUZZLES.")

print("\n" + "=" * 100)
print("CHECK 7: TIME vs TIER CONSISTENCY")
print("=" * 100)
print("Paste full run log text into log_text below to enable this check.")

log_text = """
PASTE_FULL_RUN_LOG_HERE
"""

times = {}
for match in re.finditer(r"finished (\w+) in ([\d.]+)s", log_text):
    times[match.group(1)] = float(match.group(2))

if times:
    print(f"Found timing data for {len(times)} puzzles:")
    for k, t in sorted(times.items(), key=lambda x: x[1]):
        if k in data_dict:
            tier, _ = get_puzzle_priority(k, data_dict)
            print(f"  {k}: {t:.1f}s, tier={tier}")
else:
    print("No timing data found.")

print("\n" + "=" * 100)
print("CHECK 8: LENS-WISE SOLVE / NO-OUTPUT BREAKDOWN")
print("=" * 100)

def grid_cells(grid):
    return int(np.prod(np.shape(grid)))

def debug_feature_tags(pid, data_dict):
    puzzle = data_dict[pid]
    train = puzzle["train"]

    input_sizes = [grid_cells(ex["input"]) for ex in train]
    output_sizes = [grid_cells(ex["output"]) for ex in train]
    ratios = [
        max(i, o) / max(min(i, o), 1)
        for i, o in zip(input_sizes, output_sizes)
    ]

    colors = set()
    for ex in train:
        for field in ["input", "output"]:
            arr = np.array(ex[field])
            colors.update(np.unique(arr).tolist())

    tier, fine = get_puzzle_priority(pid, data_dict)
    max_ratio = float(max(ratios))
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and len(colors) <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")

    return {
        "tier": tier,
        "fine": fine,
        "tags": tags,
        "rare": is_rare,
        "dynamic": is_dynamic,
        "multi_query": n_queries > 1,
        "high_ratio": max_ratio > 5,
        "extreme_ratio": max_ratio > 30,
    }

feature_by_id = {
    pid: debug_feature_tags(pid, data_dict)
    for pid in DEBUG_PUZZLES
    if pid in data_dict
}

summary_by_id = {r["id"]: r for r in summary_data}

LENSES = {
    "ALL_48": lambda f: True,
    "TIER1": lambda f: f["tier"] == 1,
    "TIER2": lambda f: f["tier"] == 2,
    "TIER3": lambda f: f["tier"] == 3,
    "EASY_STATIC": lambda f: "easy_static" in f["tags"],
    "RARE": lambda f: f["rare"],
    "MULTI_QUERY": lambda f: f["multi_query"],
    "DYNAMIC": lambda f: f["dynamic"],
    "HIGH_RATIO": lambda f: f["high_ratio"],
    "EXTREME_RATIO": lambda f: f["extreme_ratio"],
}

lens_rows = []

for lens_name, lens_fn in LENSES.items():
    ids = [pid for pid, feat in feature_by_id.items() if lens_fn(feat)]

    solved_ids = [pid for pid in ids if is_solved(summary_by_id[pid])]
    no_output_ids = [pid for pid in ids if is_no_output(summary_by_id[pid])]
    unsolved_ids = [pid for pid in ids if is_unsolved(summary_by_id[pid])]
    dual_ids = [pid for pid in ids if dual_yes(summary_by_id[pid])]

    ran_n = len(ids) - len(no_output_ids)
    solve_rate = len(solved_ids) / ran_n * 100 if ran_n else 0.0
    dual_rate = len(dual_ids) / ran_n * 100 if ran_n else 0.0
    no_output_rate = len(no_output_ids) / len(ids) * 100 if ids else 0.0

    lens_rows.append({
        "lens": lens_name,
        "total": len(ids),
        "ran": ran_n,
        "solved": len(solved_ids),
        "dual": len(dual_ids),
        "unsolved": len(unsolved_ids),
        "no_output": len(no_output_ids),
        "solve_rate": solve_rate,
        "dual_rate": dual_rate,
        "no_output_rate": no_output_rate,
        "solved_ids": solved_ids,
        "unsolved_ids": unsolved_ids,
        "no_output_ids": no_output_ids,
    })

print(f"{'Lens':<15} {'Total':>5} {'Ran':>5} {'Solved':>7} {'Dual':>6} {'NoOut':>6} {'Solve%':>8} {'Dual%':>8} {'NoOut%':>8}")
print("-" * 90)

for r in lens_rows:
    print(
        f"{r['lens']:<15} {r['total']:>5} {r['ran']:>5} "
        f"{r['solved']:>7} {r['dual']:>6} {r['no_output']:>6} "
        f"{r['solve_rate']:>7.1f}% {r['dual_rate']:>7.1f}% {r['no_output_rate']:>7.1f}%"
    )

print("\nLens failure details:")
for r in lens_rows:
    if r["unsolved_ids"] or r["no_output_ids"]:
        print(f"\n{r['lens']}:")
        print(f"  unsolved : {r['unsolved_ids']}")
        print(f"  no_output: {r['no_output_ids']}")

print("\n" + "=" * 100)
print("FINAL DECISION GUARD FOR MAIN_DEBUG_48_EVAL_PROXY")
print("=" * 100)

lens_map = {r["lens"]: r for r in lens_rows}

print("Critical lenses to protect:")
for name in ["ALL_48", "TIER1", "EASY_STATIC"]:
    r = lens_map[name]
    print(
        f"  {name:<12}: solved={r['solved']}/{r['ran']} "
        f"dual={r['dual']}/{r['ran']} no_output={r['no_output']}/{r['total']}"
    )

print("\nWatch lenses:")
for name in ["RARE", "MULTI_QUERY", "DYNAMIC", "TIER3"]:
    r = lens_map[name]
    print(
        f"  {name:<12}: solved={r['solved']}/{r['ran']} "
        f"dual={r['dual']}/{r['ran']} no_output={r['no_output']}/{r['total']}"
    )

print("\nDecision rule:")
print("  KEEP if ALL_48 improves AND TIER1 does not drop AND EASY_STATIC does not drop AND no_output does not increase.")
print("  HOLD/REJECT if gain comes mainly from RARE/MULTI_QUERY while TIER1 or EASY_STATIC drops.")
print("  HOLD/REJECT if no_output increases, even when solved count improves.")

print("\n" + "=" * 100)
print("FINAL SUMMARY: TIER vs SOLVE RATE")
print("=" * 100)

tier_solve = {
    1: {"total": 0, "solved": 0},
    2: {"total": 0, "solved": 0},
    3: {"total": 0, "solved": 0},
}

for row in summary_data:
    if is_no_output(row):
        continue
    if row["id"] in data_dict:
        tier, _ = get_puzzle_priority(row["id"], data_dict)
        tier_solve[tier]["total"] += 1
        if is_solved(row):
            tier_solve[tier]["solved"] += 1

for tier in [1, 2, 3]:
    total_t = tier_solve[tier]["total"]
    solved_t = tier_solve[tier]["solved"]
    if total_t > 0:
        rate = solved_t / total_t * 100
        print(f"Tier {tier}: {solved_t}/{total_t} solved ({rate:.1f}%)")
    else:
        print(f"Tier {tier}: no ran puzzles in debug set")

print("=" * 100)
print("DIAGNOSTIC COMPLETE.")
print("=" * 100)